# {title}

状态：**{status}**。单句生成用于体验控制方式，不代表标签准确率或稳定性能。

[官方代码]({source_url}) · [模型权重]({model_url}) · 许可：{license}。

本例在文本前添加括号内音色描述；生成结果的音色、情绪需人工或独立模型复核。


## 1. 检查环境
建议 GPU，预留至少 12 GiB 磁盘。

In [ ]:
import platform, shutil, subprocess, sys
from pathlib import Path
assert Path('/content').exists(), '请在 Google Colab 中运行'
assert shutil.which('nvidia-smi'), '请先选择 GPU 运行时'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
print('空闲磁盘 GiB:', round(shutil.disk_usage('/content').free / 2**30, 2))
assert shutil.disk_usage('/content').free / 2**30 >= 12


## 2. 安装官方源码

In [ ]:
SOURCE_REVISION = '300e4260b509066151d623dc9c0aa922cf82232b'
SOURCE_DIR = Path('/content/VoxCPM')
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/OpenBMB/VoxCPM.git', str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', 'origin', SOURCE_REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', SOURCE_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), 'huggingface_hub'], check=True)
print('源码:', SOURCE_REVISION)


## 3. 下载固定模型并加载
权重下载到 `/content/models/voxcpm2`。

In [ ]:
import os
os.environ['HF_HOME'] = '/content/models/.hf-cache'
from huggingface_hub import snapshot_download
MODEL_DIR = Path('/content/models/voxcpm2')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REVISION = '{model_revision}'
snapshot_download(repo_id='{model_name}', revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
print('模型:', MODEL_DIR, MODEL_REVISION)
import hashlib
for name in ('model.safetensors',):
    path = MODEL_DIR / name
    if path.exists():
        with path.open('rb') as f:
            digest = hashlib.file_digest(f, 'sha256').hexdigest()
        print('SHA-256:', name, path.stat().st_size, digest)
import importlib.metadata as metadata
import torch
from voxcpm import VoxCPM
for pkg in ('voxcpm', 'torch', 'transformers', 'huggingface_hub'):
    print(pkg, metadata.version(pkg))
print('CUDA:', torch.version.cuda, torch.cuda.get_device_name(0))
model = VoxCPM.from_pretrained(str(MODEL_DIR), load_denoiser=False, optimize=False)


## 4. 输入音色描述与文本
`TEXT` 前的括号描述会进入模型输入。计时仅包含单次生成调用与 CUDA 同步。

In [ ]:
import time, soundfile as sf
VOICE_DESCRIPTION = 'A gentle adult female voice, calm and cheerful'
TEXT = '今天的阳光很好，我们一起去散步吧。'
MODEL_INPUT = f'({VOICE_DESCRIPTION}){TEXT}'
OUTPUT_PATH = Path('/content/voxcpm2_voice_design.wav')
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
wav = model.generate(text=MODEL_INPUT, cfg_value=2.0, inference_timesteps=10, seed=42)
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
sample_rate = model.tts_model.sample_rate
sf.write(str(OUTPUT_PATH), wav, sample_rate)
print('输入:', MODEL_INPUT)
print('采样率:', sample_rate, '音频时长 s:', round(len(wav) / sample_rate, 2))
print('生成耗时 s:', round(elapsed, 3))
print('GPU 峰值 allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU 峰值 reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))


In [ ]:
from IPython.display import Audio, display
from google.colab import files
print('输出:', OUTPUT_PATH, '字节:', OUTPUT_PATH.stat().st_size)
display(Audio(filename=str(OUTPUT_PATH)))
files.download(str(OUTPUT_PATH))


## 结果边界
状态：**{status}**。描述只是生成条件，输出是否符合描述仍需验证。